# CloudFormation stacks

Until now every AWS resource was created by a command in a notebook. Here the same resources are described in three
YAML files (`infra/cloudformation/`) and CloudFormation creates them. A **stack** is one deployed YAML file:
change the file, deploy again, and CloudFormation changes only what differs. Delete the stack and its resources go too.

| Stack | File | Holds |
|---|---|---|
| `pf-cicd` | `github_oidc.yaml` | the role GitHub logs in as, and the role CloudFormation works as |
| `pf-foundation` | `foundation.yaml` | buckets, catalog databases, Athena workgroup, the four service roles |
| `pf-pipeline` | `pipeline.yaml` | ingest Lambda, Glue job, Step Functions workflow, EventBridge rule |

The notebook checks the templates, **retires the hand-built resources** (CloudFormation cannot create a name that already
exists), deploys the three stacks, checks the result, and loads the data again.

## 1. Setup

In [1]:
import json
import os
import subprocess
import sys
from pathlib import Path

import boto3

# Works whether Jupyter starts in the project root or in notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "infra"))

# deploy.py and stacks.py are the same code GitHub Actions uses; aws() runs the AWS CLI (see infra/aws_helpers.py)
import deploy
import stacks
import teardown
from aws_helpers import LAKE, batches_in_landing, REGION, RESULTS, SETTINGS, account_id, aws, aws_json, run_athena, run_sql_file, whoami

CONFIG = deploy.load_config()
SESSION = boto3.Session(region_name=CONFIG["region"])
NAMES = stacks.stack_names(CONFIG["project_prefix"])
CICD_STACK = f"{CONFIG['project_prefix']}-cicd"
TEMPLATES = ROOT / "infra" / "cloudformation"
VERSION = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, capture_output=True, text=True).stdout.strip() or "local"

print("identity:", whoami())
print("region  :", REGION)
print("stacks  :", list(NAMES.values()) + [CICD_STACK])
print("version :", VERSION, "(names this deployment; GitHub uses the full commit id)")

identity: user/Ahmed
region  : us-east-1
stacks  : ['pf-foundation', 'pf-pipeline', 'pf-cicd']
version : 038e416 (names this deployment; GitHub uses the full commit id)


## 2. Check the templates
`cfn-lint` finds wrong property names and broken references before AWS sees the file. `validate-template` is AWS's own syntax check.

In [2]:
lint = subprocess.run([str(Path(sys.executable).parent / "cfn-lint"), *map(str, sorted(TEMPLATES.glob("*.yaml")))],
                      capture_output=True, text=True)
print("cfn-lint:", "clean" if lint.returncode == 0 else lint.stdout + lint.stderr)
assert lint.returncode == 0

for template in sorted(TEMPLATES.glob("*.yaml")):
    result = aws_json("cloudformation", "validate-template", "--template-body", f"file://{template.relative_to(ROOT).as_posix()}", "--output", "json")
    print(f"OK  {template.name}: AWS accepts it; parameters {[p['ParameterKey'] for p in result['Parameters']]}")

cfn-lint: clean


OK  foundation.yaml: AWS accepts it; parameters ['ResultsBucketName', 'LakeBucketName', 'ProjectPrefix']


OK  github_oidc.yaml: AWS accepts it; parameters ['ResultsBucketName', 'GitHubSubject', 'LakeBucketName', 'OidcProviderArn', 'ProjectPrefix']


OK  pipeline.yaml: AWS accepts it; parameters ['NumberOfBatches', 'FoundationStackName', 'FirstIngestDate', 'DeploymentVersion', 'ProjectPrefix']


## 3. Retire the hand-built resources
The resources made in Phases 3-10 have the names the templates use. They are deleted first, in a safe order
(trigger, workflow, jobs, workgroup, catalog, roles, buckets). **The data goes too**; Section 9 loads it again from `data/landing/`.

This cell deletes only when it is told to (`PF_CONFIRM_RETIRE=yes` in the environment) **and** the foundation stack does not exist yet,
so running the notebook again later can never delete what CloudFormation owns.

In [3]:
plan = teardown.hand_built_resources(CONFIG)
print("hand-built resources, in deletion order:")
for kind, name in plan:
    print(f"  {kind:<14}{name}")

stack_exists = bool(aws("cloudformation", "describe-stacks", "--stack-name", NAMES["foundation"], check=False, show=False).returncode == 0)
if stack_exists:
    print("\nOK  the foundation stack exists: the hand-built resources are already retired, nothing to do")
elif os.environ.get("PF_CONFIRM_RETIRE") == "yes":
    teardown.delete_hand_built(SESSION, CONFIG)
    print("\nOK  hand-built resources retired")
else:
    raise SystemExit("not deleting: set PF_CONFIRM_RETIRE=yes to delete the list above")

hand-built resources, in deletion order:
  rule          pf-new-batch-rule
  state_machine pf-batch-pipeline
  glue_job      pf-bronze-to-silver
  lambda        pf-ingest
  log_group     /aws/lambda/pf-ingest
  log_group     /aws/vendedlogs/states/pf-batch-pipeline
  workgroup     pf-wg
  glue_database pf_gold
  glue_database pf_silver
  role          pf-trigger-role
  role          pf-step-functions-role
  role          pf-glue-role
  role          pf-lambda-role
  role          pf-github-deploy-role
  bucket        patientflow-athena-query-results-zhmkbf
  bucket        patientflow-datalake-zhmkbf


$ aws cloudformation describe-stacks --stack-name pf-foundation


deleted  EventBridge rule pf-new-batch-rule


deleted  state machine pf-batch-pipeline


deleted  Glue job pf-bronze-to-silver


...

deleted  IAM role pf-github-deploy-role


         1047 objects removed from patientflow-athena-query-results-zhmkbf


deleted  S3 bucket patientflow-athena-query-results-zhmkbf


         4389 objects removed from patientflow-datalake-zhmkbf


deleted  S3 bucket patientflow-datalake-zhmkbf

OK  hand-built resources retired


## 4. The CI/CD stack
Deployed by you (an administrator), once. It creates the **deploy role** (what GitHub logs in as; it can only hand work to
CloudFormation and upload code) and the **CloudFormation role** (the only one allowed to create the pipeline's resources, and only those names).
The GitHub login provider already exists in the account (an account can have one), so its ARN is passed in.

In [4]:
provider = next(p["Arn"] for p in aws_json("iam", "list-open-id-connect-providers")["OpenIDConnectProviderList"]
                if p["Arn"].endswith("token.actions.githubusercontent.com"))
CICD_PARAMETERS = {
    "ProjectPrefix": CONFIG["project_prefix"], "LakeBucketName": CONFIG["lake_bucket"],
    "ResultsBucketName": CONFIG["results_bucket"], "GitHubSubject": SETTINGS["github_subject"],
    "OidcProviderArn": provider,
}
command = stacks.deploy_command(CICD_STACK, TEMPLATES / "github_oidc.yaml", CICD_PARAMETERS, CONFIG["region"])
result = subprocess.run(command, capture_output=True, text=True)
print((result.stdout + result.stderr).strip().replace(account_id(), "<account>")[-400:])
assert result.returncode == 0

outputs = {o["OutputKey"]: o["OutputValue"] for o in
           aws_json("cloudformation", "describe-stacks", "--stack-name", CICD_STACK)["Stacks"][0]["Outputs"]}
CFN_ROLE_ARN = outputs["CloudFormationRoleArn"]
print("deploy role         :", outputs["DeployRoleArn"].replace(account_id(), "<account>"))
print("cloudformation role :", CFN_ROLE_ARN.replace(account_id(), "<account>"))

Waiting for changeset to be created..
Waiting for stack create/update to complete
Successfully created/updated stack - pf-cicd


deploy role         : arn:aws:iam::<account>:role/pf-github-deploy-role
cloudformation role : arn:aws:iam::<account>:role/pf-cloudformation-role


## 5. The foundation and pipeline stacks
Same three steps GitHub Actions runs (`infra/stacks.py`), here as the CloudFormation role:
deploy the foundation stack, upload the code of this version to the lake bucket, deploy the pipeline stack that points at that code.

In [5]:
def deploy_stack(name, template, parameters):
    command = stacks.deploy_command(name, TEMPLATES / template, parameters, CONFIG["region"], CFN_ROLE_ARN)
    result = subprocess.run(command, capture_output=True, text=True)
    print((result.stdout + result.stderr).strip().replace(account_id(), "<account>")[-300:])
    assert result.returncode == 0, result.stderr


deploy_stack(NAMES["foundation"], "foundation.yaml", stacks.foundation_parameters(CONFIG))

Waiting for changeset to be created..
Waiting for stack create/update to complete
Successfully created/updated stack - pf-foundation


In [6]:
built = stacks.build_artifacts(CONFIG, account_id(), VERSION, ROOT / "build")
uploaded = stacks.upload_artifacts(SESSION, CONFIG, VERSION, built)
assert len(uploaded) == 4

uploaded s3://patientflow-datalake-zhmkbf/glue_scripts/038e416/bronze_to_silver.py


uploaded s3://patientflow-datalake-zhmkbf/glue_scripts/038e416/fhir_utils.py


uploaded s3://patientflow-datalake-zhmkbf/artifacts/038e416/ingest.zip


uploaded s3://patientflow-datalake-zhmkbf/artifacts/038e416/state_machine.json


In [7]:
deploy_stack(NAMES["pipeline"], "pipeline.yaml", stacks.pipeline_parameters(CONFIG, VERSION))

Waiting for changeset to be created..
Waiting for stack create/update to complete
Successfully created/updated stack - pf-pipeline


## 6. What CloudFormation built

In [8]:
for stack in (CICD_STACK, NAMES["foundation"], NAMES["pipeline"]):
    description = aws_json("cloudformation", "describe-stacks", "--stack-name", stack)["Stacks"][0]
    print(f"{stack:<16}{description['StackStatus']}")
    assert description["StackStatus"] in ("CREATE_COMPLETE", "UPDATE_COMPLETE")
    resources = aws_json("cloudformation", "describe-stack-resources", "--stack-name", stack)["StackResources"]
    for resource in resources:
        print(f"    {resource['ResourceType']:<30}{resource['LogicalResourceId']:<22}{resource['ResourceStatus']}")

pf-cicd         CREATE_COMPLETE


    AWS::IAM::Role                CloudFormationRole    CREATE_COMPLETE
    AWS::IAM::Role                DeployRole            CREATE_COMPLETE


pf-foundation   CREATE_COMPLETE


    AWS::IAM::Role                GlueRole              CREATE_COMPLETE
    AWS::Glue::Database           GoldDatabase          CREATE_COMPLETE
    AWS::S3::Bucket               LakeBucket            CREATE_COMPLETE
    AWS::IAM::Role                LambdaRole            CREATE_COMPLETE
    AWS::S3::Bucket               ResultsBucket         CREATE_COMPLETE
    AWS::Glue::Database           SilverDatabase        CREATE_COMPLETE
    AWS::IAM::Role                StepFunctionsRole     CREATE_COMPLETE
    AWS::IAM::Role                TriggerRole           CREATE_COMPLETE
    AWS::Athena::WorkGroup        Workgroup             CREATE_COMPLETE


pf-pipeline     CREATE_COMPLETE


    AWS::Glue::Job                BronzeToSilverJob     CREATE_COMPLETE
    AWS::Lambda::Function         IngestFunction        CREATE_COMPLETE
    AWS::Logs::LogGroup           IngestLogGroup        CREATE_COMPLETE
    AWS::Events::Rule             NewBatchRule          CREATE_COMPLETE
    AWS::StepFunctions::StateMachineWorkflow              CREATE_COMPLETE
    AWS::Logs::LogGroup           WorkflowLogGroup      CREATE_COMPLETE


## 7. Do the resources match what the notebooks built by hand?
The same checks as Phases 3, 4, 5 and 10.

In [9]:
# --- buckets (Phase 3)
for bucket in (LAKE, RESULTS):
    block = aws_json("s3api", "get-public-access-block", "--bucket", bucket)["PublicAccessBlockConfiguration"]
    assert all(block.values()), bucket
    algorithm = aws_json("s3api", "get-bucket-encryption", "--bucket", bucket)["ServerSideEncryptionConfiguration"]["Rules"][0][
        "ApplyServerSideEncryptionByDefault"]["SSEAlgorithm"]
    assert algorithm == "AES256"
    print(f"OK  {bucket}: public access blocked, encryption {algorithm}")
assert aws_json("s3api", "get-bucket-lifecycle-configuration", "--bucket", RESULTS)["Rules"][0]["Expiration"]["Days"] == 7
assert "EventBridgeConfiguration" in aws_json("s3api", "get-bucket-notification-configuration", "--bucket", LAKE)
print("OK  results expire after 7 days; the lake bucket sends its events to EventBridge")

# --- catalog and workgroup (Phase 3)
databases = {d["Name"] for d in aws_json("glue", "get-databases")["DatabaseList"]}
assert {"pf_silver", "pf_gold"} <= databases
workgroup = aws_json("athena", "get-work-group", "--work-group", SETTINGS["athena_workgroup"])["WorkGroup"]["Configuration"]
assert workgroup["EnforceWorkGroupConfiguration"] and workgroup["BytesScannedCutoffPerQuery"] == 1024 ** 3
print("OK  catalog databases and the workgroup (enforced, 1 GB scan limit)")

# --- Lambda (Phase 4) and Glue job (Phase 5)
function = aws_json("lambda", "get-function-configuration", "--function-name", SETTINGS["lambda_name"])
assert (function["Runtime"], function["Timeout"], function["MemorySize"], function["State"]) == ("python3.12", 60, 256, "Active")
assert function["Environment"]["Variables"]["LAKE_BUCKET"] == LAKE
job = aws_json("glue", "get-job", "--job-name", SETTINGS["glue_job_name"])["Job"]
assert (job["GlueVersion"], job["WorkerType"], job["NumberOfWorkers"], job["Timeout"], job["MaxRetries"]) == ("5.0", "G.1X", 2, 10, 0)
assert job["Command"]["ScriptLocation"].endswith(f"glue_scripts/{VERSION}/bronze_to_silver.py")
print(f"OK  Lambda and Glue job; the Glue job reads the scripts of version {VERSION}")

# --- workflow and trigger (Phases 7, 10)
machine_arn = f"arn:aws:states:{REGION}:{account_id()}:stateMachine:{SETTINGS['state_machine_name']}"
machine = aws_json("stepfunctions", "describe-state-machine", "--state-machine-arn", machine_arn)
assert machine["loggingConfiguration"]["level"] == "ERROR" and "DataQualityChecks" in json.loads(machine["definition"])["States"]
rule = aws_json("events", "describe-rule", "--name", SETTINGS["trigger_rule_name"])
targets = aws_json("events", "list-targets-by-rule", "--rule", SETTINGS["trigger_rule_name"])["Targets"]
assert rule["State"] == "ENABLED" and len(targets) == 1 and targets[0]["Arn"] == machine_arn
print("OK  workflow (with the data-quality steps, error logging) and the trigger rule (one target: the workflow)")

OK  patientflow-datalake-zhmkbf: public access blocked, encryption AES256


OK  patientflow-athena-query-results-zhmkbf: public access blocked, encryption AES256


OK  results expire after 7 days; the lake bucket sends its events to EventBridge


OK  catalog databases and the workgroup (enforced, 1 GB scan limit)


OK  Lambda and Glue job; the Glue job reads the scripts of version 038e416


OK  workflow (with the data-quality steps, error logging) and the trigger rule (one target: the workflow)


## 8. Permission checks
The AWS policy simulator answers "would this role be allowed to ...?" without doing it.

In [10]:
def decision(role, action, resource, context=None):
    arn = aws_json("iam", "get-role", "--role-name", role)["Role"]["Arn"]
    args = ["iam", "simulate-principal-policy", "--policy-source-arn", arn, "--action-names", action, "--resource-arns", resource]
    if context:
        args += ["--context-entries", f"ContextKeyName=iam:PassedToService,ContextKeyValues={context},ContextKeyType=string"]
    return aws_json(*args, "--query", "EvaluationResults[0].EvalDecision", "--output", "json")


A, R, P = account_id(), REGION, CONFIG["project_prefix"]
cfn_role_arn = f"arn:aws:iam::{A}:role/{P}-cloudformation-role"
CHECKS = [
    # the service roles of the pipeline (same expectations as the iam_roles notebook)
    (f"{P}-lambda-role", "s3:GetObject", f"arn:aws:s3:::{LAKE}/landing/batch_01/a.json", "allowed"),
    (f"{P}-lambda-role", "s3:PutObject", f"arn:aws:s3:::{LAKE}/silver/a.parquet", "implicitDeny"),
    (f"{P}-glue-role", "s3:PutObject", f"arn:aws:s3:::{LAKE}/silver/a.parquet", "allowed"),
    (f"{P}-glue-role", "s3:PutObject", f"arn:aws:s3:::{LAKE}/bronze/patient_records/a.json", "implicitDeny"),
    (f"{P}-step-functions-role", "glue:StartJobRun", f"arn:aws:glue:{R}:{A}:job/{SETTINGS['glue_job_name']}", "allowed"),
    (f"{P}-trigger-role", "states:StartExecution", machine_arn, "allowed"),
    (f"{P}-trigger-role", "states:DeleteStateMachine", machine_arn, "implicitDeny"),
    # the deploy role: it hands work to CloudFormation, it cannot build anything itself
    (f"{P}-github-deploy-role", "cloudformation:UpdateStack", f"arn:aws:cloudformation:{R}:{A}:stack/{P}-pipeline/x", "allowed"),
    (f"{P}-github-deploy-role", "cloudformation:DeleteStack", f"arn:aws:cloudformation:{R}:{A}:stack/{P}-pipeline/x", "implicitDeny"),
    (f"{P}-github-deploy-role", "s3:PutObject", f"arn:aws:s3:::{LAKE}/artifacts/x/ingest.zip", "allowed"),
    (f"{P}-github-deploy-role", "s3:PutObject", f"arn:aws:s3:::{LAKE}/gold/a.parquet", "implicitDeny"),
    (f"{P}-github-deploy-role", "lambda:CreateFunction", f"arn:aws:lambda:{R}:{A}:function:{P}-ingest", "implicitDeny"),
    (f"{P}-github-deploy-role", "iam:CreateRole", f"arn:aws:iam::{A}:role/anything", "implicitDeny"),
    # the CloudFormation role: only the project's own names
    (f"{P}-cloudformation-role", "lambda:CreateFunction", f"arn:aws:lambda:{R}:{A}:function:{P}-ingest", "allowed"),
    (f"{P}-cloudformation-role", "lambda:CreateFunction", f"arn:aws:lambda:{R}:{A}:function:somebody-else", "implicitDeny"),
    (f"{P}-cloudformation-role", "iam:CreateRole", f"arn:aws:iam::{A}:role/{P}-glue-role", "allowed"),
    (f"{P}-cloudformation-role", "iam:CreateRole", f"arn:aws:iam::{A}:role/admin-backdoor", "implicitDeny"),
    (f"{P}-cloudformation-role", "iam:AttachRolePolicy", f"arn:aws:iam::{A}:role/{P}-glue-role", "implicitDeny"),
]
for role, action, resource, expected in CHECKS:
    got = decision(role, action, resource)
    assert got == expected, f"{role} {action}: expected {expected}, got {got}"
    print(f"OK  {role:<26}{action:<30}{expected}")

passed = decision(f"{P}-github-deploy-role", "iam:PassRole", cfn_role_arn, context="cloudformation.amazonaws.com")
other = decision(f"{P}-github-deploy-role", "iam:PassRole", cfn_role_arn, context="lambda.amazonaws.com")
assert (passed, other) == ("allowed", "implicitDeny")
print("OK  the deploy role may pass the CloudFormation role to CloudFormation only")

OK  pf-lambda-role            s3:GetObject                  allowed


OK  pf-lambda-role            s3:PutObject                  implicitDeny


OK  pf-glue-role              s3:PutObject                  allowed


OK  pf-glue-role              s3:PutObject                  implicitDeny


OK  pf-step-functions-role    glue:StartJobRun              allowed


...

OK  pf-cloudformation-role    lambda:CreateFunction         implicitDeny


OK  pf-cloudformation-role    iam:CreateRole                allowed


OK  pf-cloudformation-role    iam:CreateRole                implicitDeny


OK  pf-cloudformation-role    iam:AttachRolePolicy          implicitDeny


OK  the deploy role may pass the CloudFormation role to CloudFormation only


## 9. Load the data again
The buckets are new and empty. Upload the four batches (without markers: the markers are what starts a run),
then create the gold tables and the five KPI views. Run `run_batches.ipynb` afterwards to load all four batches through the trigger.

In [11]:
aws("s3", "sync", "data/landing", f"s3://{LAKE}/landing/", "--only-show-errors")
for batch in batches_in_landing():
    objects = aws_json("s3api", "list-objects-v2", "--bucket", LAKE, "--prefix", f"landing/batch_{batch:02d}/").get("Contents", [])
    print(f"batch {batch}: {len(objects)} files in landing/")
    assert objects and not any(o["Key"].endswith("_SUCCESS") for o in objects)

$ aws s3 sync data/landing s3://patientflow-datalake-zhmkbf/landing/ --only-show-errors


batch 1: 37 files in landing/


batch 2: 34 files in landing/


batch 3: 34 files in landing/


batch 4: 34 files in landing/


In [12]:
existing = {row[0] for row in run_athena("SHOW TABLES IN pf_gold")["rows"]}
for table in ("dim_patient", "dim_facility", "dim_practitioner", "fact_encounter", "dim_date"):
    if table == "dim_date" and "dim_date" in existing:
        print("OK  table dim_date exists")
        continue
    run_sql_file(f"sql/gold/ddl/{table}.sql")
    print("OK  table", table)
for path in sorted((ROOT / "sql" / "gold" / "views").glob("*.sql")):
    run_sql_file(f"sql/gold/views/{path.name}")
    print("OK  view ", path.stem)

OK  table dim_patient


OK  table dim_facility


OK  table dim_practitioner


OK  table fact_encounter


OK  table dim_date


OK  view  vw_avg_los_by_facility_month


OK  view  vw_daily_admissions_discharges


OK  view  vw_ed_volume_by_hour_dow


OK  view  vw_readmission_30d


OK  view  vw_top_reasons


## What this is for
| Item | Where |
|---|---|
| Change the infrastructure | edit a YAML file in `infra/cloudformation/`, push to `main`: GitHub deploys it (`deploy.yml`) |
| See what a stack holds | AWS console > CloudFormation > Stacks > Resources / Events / Outputs |
| Load the data on the new resources | `run_batches.ipynb`, or the `Release batch` workflow on GitHub |
| Earlier notebooks (`s3_buckets`, `iam_roles`, `ingest_lambda`, ...) | the step-by-step manual build, kept for learning. They must not be run again against the stacks (they would change resources outside CloudFormation) |
| Delete everything | `infra/teardown.py` (Phase 14) |